In [1]:
install.packages("ggraph")

library(igraph)
library(ggraph)
library(dplyr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘tweenr’, ‘polyclip’, ‘gridExtra’, ‘RcppArmadillo’, ‘ggforce’, ‘igraph’, ‘ggrepel’, ‘viridis’, ‘tidygraph’, ‘graphlayouts’


Warning message in install.packages("ggraph"):
“installation of package ‘graphlayouts’ had non-zero exit status”
Warning message in install.packages("ggraph"):
“installation of package ‘ggraph’ had non-zero exit status”

Attaching package: ‘igraph’


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union




ERROR: Error in library(ggraph): there is no package called ‘ggraph’


In [ ]:
# --- OPTION A: If you have an .RData or .rda file ---
# load("congress.RData")

# --- OPTION B: If you have CSV files ---
common_committees <- read.csv("common_committees.csv")
congress_members <- read.csv("congress_members.csv")

# Inspect column names so we use the exact variable names in the next steps
head(common_committees)
head(congress_members)

In [ ]:
# Create an undirected igraph from common_committees
# (If you don't have a separate congress_members dataframe, remove the 'vertices =' line)
g_all <- graph_from_data_frame(
  d = common_committees,
  vertices = congress_members,
  directed = FALSE
)

# Print number of vertices and edges
cat("Total Vertices:", vcount(g_all), "\n")
cat("Total Edges:", ecount(g_all), "\n")

In [ ]:
# Filter edges where shared committees >= 3
common_committees_3 <- common_committees %>%
  filter(committees >= 3)

g_filtered <- graph_from_data_frame(
  d = common_committees_3,
  vertices = congress_members,
  directed = FALSE
)

cat("Number of edges (at least 3 committees):", ecount(g_filtered), "\n")

In [ ]:
# Set plot dimensions for Colab output
options(repr.plot.width = 10, repr.plot.height = 8)

ggraph(g_filtered, layout = "fr") +
  geom_edge_link(alpha = 0.4, color = "gray65") +
  geom_node_point(size = 3, color = "steelblue") +
  theme_void() +
  labs(title = "Congress Members on at Least 3 Committees Together")

In [ ]:
# Check exact values in the chamber column
table(V(g_all)$chamber)

# Keep only Senators (change "Senate" to "Senator" if your table output shows "Senator")
senator_nodes <- V(g_all)[chamber == "Senate"]

g_senators <- induced_subgraph(g_all, vids = senator_nodes)

cat("Number of Senators (vertices):", vcount(g_senators), "\n")
cat("Number of edges among Senators:", ecount(g_senators), "\n")

# Visualize the Senate-only network
ggraph(g_senators, layout = "fr") +
  geom_edge_link(alpha = 0.35, color = "gray60") +
  geom_node_point(size = 3.5, color = "darkred") +
  theme_void() +
  labs(title = "US Senate Committee Co-Membership Network")

In [ ]:
install.packages("ggraph")

library(igraph)
library(ggraph)
library(dplyr)

In [ ]:
set.seed(42)

# 1. Create vertex dataframe (Congress members with chamber and party attributes)
congress_members <- data.frame(
  name = c(
    "Sen_Smith", "Sen_Johnson", "Sen_Williams", "Sen_Brown", "Sen_Jones",
    "Sen_Garcia", "Sen_Miller", "Rep_Davis", "Rep_Rodriguez", "Rep_Martinez",
    "Rep_Hernandez", "Rep_Lopez", "Rep_Gonzalez", "Rep_Wilson", "Rep_Anderson"
  ),
  chamber = c(
    "Senate", "Senate", "Senate", "Senate", "Senate",
    "Senate", "Senate", "House", "House", "House",
    "House", "House", "House", "House", "House"
  ),
  party = c(
    "Democrat", "Republican", "Democrat", "Republican", "Democrat",
    "Democrat", "Republican", "Republican", "Democrat", "Republican",
    "Democrat", "Democrat", "Republican", "Republican", "Democrat"
  ),
  stringsAsFactors = FALSE
)

# 2. Create edge dataframe (common_committees) between pairs of members
all_pairs <- t(combn(congress_members$name, 2))
common_committees <- data.frame(
  from = all_pairs[, 1],
  to = all_pairs[, 2],
  committees = sample(0:5, nrow(all_pairs), replace = TRUE, prob = c(0.45, 0.2, 0.15, 0.1, 0.06, 0.04)),
  stringsAsFactors = FALSE
) %>%
  filter(committees > 0) # Keep pairs that share at least 1 committee

# Preview the generated dataframes
head(congress_members)
head(common_committees)

In [ ]:
g_all <- graph_from_data_frame(
  d = common_committees,
  vertices = congress_members,
  directed = FALSE
)

cat("Number of vertices:", vcount(g_all), "\n")
cat("Number of edges:", ecount(g_all), "\n")

In [ ]:
common_committees_3 <- common_committees %>%
  filter(committees >= 3)

g_filtered <- graph_from_data_frame(
  d = common_committees_3,
  vertices = congress_members,
  directed = FALSE
)

cat("Number of edges (at least 3 committees):", ecount(g_filtered), "\n")

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 7)

ggraph(g_filtered, layout = "fr") +
  geom_edge_link(aes(width = committees), alpha = 0.5, color = "gray60") +
  geom_node_point(aes(color = chamber), size = 5) +
  geom_node_text(aes(label = name), repel = TRUE, size = 3.5) +
  scale_edge_width(range = c(0.5, 2.5)) +
  theme_void() +
  labs(
    title = "Congress Members Sharing at Least 3 Committees",
    color = "Chamber",
    edge_width = "Shared Committees"
  )

In [ ]:
# Select only vertices where chamber is "Senate"
senator_vertices <- V(g_all)[chamber == "Senate"]

# Create induced subgraph containing only Senators
g_senators <- induced_subgraph(g_all, vids = senator_vertices)

cat("Number of Senators (vertices):", vcount(g_senators), "\n")
cat("Number of edges among Senators:", ecount(g_senators), "\n")

# Visualize the Senate-only network
ggraph(g_senators, layout = "fr") +
  geom_edge_link(aes(width = committees), alpha = 0.5, color = "gray60") +
  geom_node_point(aes(color = party), size = 5) +
  geom_node_text(aes(label = name), repel = TRUE, size = 3.5) +
  scale_color_manual(values = c("Democrat" = "steelblue", "Republican" = "firebrick")) +
  scale_edge_width(range = c(0.5, 2.5)) +
  theme_void() +
  labs(
    title = "US Senate Committee Co-Membership Network (Senators Only)",
    color = "Party",
    edge_width = "Shared Committees"
  )